# 06 — BOCPD Aggregate Across Channels (Goal 10)

Bayesian Online Change-Point Detection with **channel aggregation**. Instead of using a single channel (as in 02), we aggregate log bandpower **across all 17 channels** (mean) per repetition. This reduces channel-specific noise and yields a single BOCPD sequence per condition × band.

**Goal 10** from results.md. Reference: Adams & MacKay (2007); `refrences/deep-research-report.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from scipy.stats import norm

def logsumexp(x):
    m = np.max(x)
    return m + np.log(np.sum(np.exp(x - m)))

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else (Path.cwd() / ".." / "..").resolve()
FEATURES_DIR = ROOT / "artifacts" / "data" / "features__log_bandpower__2026-02-27"
ARTIFACTS_FIG = ROOT / "artifacts" / "figures" / "bocpd"
ARTIFACTS_TABLES = ROOT / "artifacts" / "tables"
ARTIFACTS_FIG.mkdir(parents=True, exist_ok=True)
ARTIFACTS_TABLES.mkdir(parents=True, exist_ok=True)
DATE_TAG = "2026-02-27"

## BOCPD implementation (Gaussian, unknown mean)

Constant hazard H(r)=h. Same as 02.

In [ ]:
def bocpd_gaussian(x, hazard=0.02, mean0=None, var0=1.0, varx=None):
    """
    BOCPD for scalar Gaussian: x|μ~N(μ,varx), μ~N(mean0,var0).
    Returns: R (run-length posterior), cp_prob (changepoint prob per t).
    """
    T = len(x)
    if mean0 is None:
        mean0 = np.mean(x)
    if varx is None:
        varx = np.var(x) + 1e-6
    prec0 = 1.0 / var0
    precx = 1.0 / varx
    log_H = np.log(hazard)
    log_1mH = np.log(1 - hazard)
    R = np.zeros((T + 1, T + 1))
    cp_prob = np.zeros(T)
    prec_params = np.array([prec0])
    mean_params = np.array([mean0])
    log_message = np.array([0.0])
    for t in range(T):
        xt = x[t]
        post_prec = prec_params[: t + 1]
        post_var = 1.0 / post_prec + varx
        post_std = np.sqrt(post_var)
        log_preds = norm.logpdf(xt, mean_params[: t + 1], post_std)
        log_growth = log_preds + log_message + log_1mH
        log_cp = log_preds[0] + log_H + logsumexp(log_message)
        new_log_joint = np.append(log_cp, log_growth)
        new_log_joint -= logsumexp(new_log_joint)
        R[t, : t + 2] = np.exp(new_log_joint)
        cp_prob[t] = np.exp(new_log_joint[0])
        new_prec = prec_params + precx
        prec_params = np.append([prec0], new_prec)
        new_mean = (mean_params * prec_params[:-1] + xt * precx) / new_prec
        mean_params = np.append([mean0], new_mean)
        log_message = new_log_joint
    return R[:T], cp_prob

## Channel aggregation

For each condition and band, take **mean log bandpower across all 17 channels** at each repetition. Sequence shape: (80,) per condition × band.

In [ ]:
def aggregate_channels(lbp, band_idx):
    """
    lbp: (n_cond, n_reps, n_ch, n_bands)
    band_idx: int (e.g., 2 for alpha)
    Returns: (n_cond, n_reps) - mean log bandpower across channels per cond × rep.
    """
    return lbp[:, :, :, band_idx].mean(axis=2)

## Load features and run BOCPD (channel-aggregated)

Use alpha band (band_idx=2) for comparison with 02. Same hazard (0.02).

In [ ]:
np.random.seed(42)
band_idx = 2  # alpha
bands = ["delta", "theta", "alpha", "beta", "gamma"]
band_name = bands[band_idx]

sub = "sub-01"
cond_idx = 50
d = np.load(FEATURES_DIR / f"{sub}.npz", allow_pickle=True)
lbp = d["test"]  # (200, 80, 17, 5)

# Channel-aggregated: mean over 17 channels
x_agg = aggregate_channels(lbp, band_idx)[cond_idx, :]  # (80,)

_, cp_prob = bocpd_gaussian(x_agg, hazard=0.02)
print(f"{sub} cond={cond_idx} band={band_name} (aggregate over 17 ch)")
print(f"Sequence length: {len(x_agg)}, mean={x_agg.mean():.3f}, std={x_agg.std():.3f}")
print(f"max_cp_prob={cp_prob.max():.3f}, peak_rep={np.argmax(cp_prob)}")

## Plot: channel-aggregated sequence, run-length posterior, changepoint probability

In [ ]:
R, cp_prob = bocpd_gaussian(x_agg, hazard=0.02)

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].plot(np.arange(80), x_agg, "k-", alpha=0.8)
axes[0].set_ylabel("Log bandpower (mean over 17 ch)")
axes[0].set_title(f"{sub} cond {cond_idx} {band_name} — across 80 reps (channel-aggregated)")
axes[0].grid(True, alpha=0.3)
im = axes[1].imshow(R.T[:, :80], aspect="auto", origin="lower", cmap="Blues",
                    extent=[0, 80, 0, R.shape[1]], vmin=0, vmax=0.3)
axes[1].set_ylabel("Run length")
axes[1].set_xlabel("Repetition")
axes[1].set_title("Run-length posterior")
plt.colorbar(im, ax=axes[1], label="Prob")
axes[2].plot(np.arange(80), cp_prob, "C1-", lw=1.5)
axes[2].axhline(0.02, color="gray", ls=":", alpha=0.7, label="hazard")
axes[2].set_ylabel("P(changepoint)")
axes[2].set_xlabel("Repetition")
axes[2].set_title("Changepoint probability")
axes[2].legend(loc="upper right")
axes[2].grid(True, alpha=0.3)
plt.tight_layout()
out_path = ARTIFACTS_FIG / f"bocpd__across_reps_aggregate_channels__{sub}_cond{cond_idx}_{band_name}__{DATE_TAG}.png"
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved {out_path.name}")

## Run on multiple conditions (sample) and aggregate — same design as 02

In [ ]:
np.random.seed(42)
cond_indices = np.random.choice(200, size=20, replace=False)
summary_rows = []
for sub in ["sub-01", "sub-05", "sub-10"]:
    d = np.load(FEATURES_DIR / f"{sub}.npz", allow_pickle=True)
    lbp = d["test"]
    for c in cond_indices:
        x = aggregate_channels(lbp, band_idx)[c, :]
        _, cp_prob = bocpd_gaussian(x, hazard=0.02)
        summary_rows.append({"participant": sub, "condition": c, "max_cp_prob": float(cp_prob.max())})

summary_df = pd.DataFrame(summary_rows)
table_path = ARTIFACTS_TABLES / f"bocpd__across_reps_aggregate_channels_summary__{DATE_TAG}.csv"
summary_df.to_csv(table_path, index=False)
print("Participant summary (mean/max max_cp_prob):")
print(summary_df.groupby("participant")["max_cp_prob"].agg(["mean", "max"]).to_string())
print(f"\nSaved {table_path.name}")

## Comparison: channel-aggregated vs single-channel (02)

Load 02 summary and compare max_cp_prob distributions.

In [ ]:
path_02 = ARTIFACTS_TABLES / "bocpd__across_reps_summary__2026-02-27.csv"
path_06 = ARTIFACTS_TABLES / f"bocpd__across_reps_aggregate_channels_summary__{DATE_TAG}.csv"

if path_02.exists() and path_06.exists():
    df_02 = pd.read_csv(path_02)
    df_06 = pd.read_csv(path_06)
    print("Single-channel (02, Pz alpha):")
    print(f"  mean max_cp_prob={df_02['max_cp_prob'].mean():.3f}, max={df_02['max_cp_prob'].max():.3f}")
    print("Channel-aggregated (06, mean over 17 ch, alpha):")
    print(f"  mean max_cp_prob={df_06['max_cp_prob'].mean():.3f}, max={df_06['max_cp_prob'].max():.3f}")
else:
    print("Run 02_bocpd_across_reps.ipynb first to generate comparison.")